In [1]:
import sys
from pathlib import Path
sys.path.append(str(Path.cwd().parent))

from src.utils import *

import pandas as pd

from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score

import lightgbm as lgbm
import xgboost as xg

import optuna as op

In [2]:
# Read data files
x, y, x_test = load_processed_data()

# Testing new feature combinations based on correlation

x["RelativeTyreLife"] = (x["TyreLife"] / (x["LapNumber"] + 1))

x_test["RelativeTyreLife"] = (x_test["TyreLife"] / (x_test["LapNumber"] + 1))

x["TyreLife_Per_Stint"] = (x["TyreLife"] / (x["Stint"] + 1))

x_test["TyreLife_Per_Stint"] = (x_test["TyreLife"] / (x_test["Stint"] + 1))

x["Stint_Per_RaceProgress"] = (x["Stint"] / (x["RaceProgress"] + 1))

x_test["Stint_Per_RaceProgress"] = (x_test["Stint"] / (x_test["RaceProgress"] + 1))

x["Degredation_Per_TyreLife"] = (x["Cumulative_Degradation"] / x["TyreLife"] + 1) 

x_test["Degredation_Per_TyreLife"] = (x_test["Cumulative_Degradation"] / x_test["TyreLife"] + 1) 


# Test train split from sklearn model selection
x_train, x_val, y_train, y_val = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

model = xg.XGBClassifier(
    n_estimators=2000,
    learning_rate=0.03,
    max_depth=6,
    min_child_weight=1,
    subsample=0.9,
    colsample_bytree=0.8,
    objective="binary:logistic",
    eval_metric="auc",
    random_state=42,
    n_jobs=-1,
    early_stopping_rounds=200
)

model.fit(
    x_train,
    y_train,
    eval_set=[(x_val,y_val)],
    verbose = 200
)

validation_prediction = generate_probability_predict(model, x_val)

generate_auc(y_val, validation_prediction)


[0]	validation_0-auc:0.92113
[200]	validation_0-auc:0.94344
[400]	validation_0-auc:0.94644
[600]	validation_0-auc:0.94793
[800]	validation_0-auc:0.94882
[1000]	validation_0-auc:0.94933
[1200]	validation_0-auc:0.94964
[1400]	validation_0-auc:0.94987
[1600]	validation_0-auc:0.95006
[1800]	validation_0-auc:0.95019
[1999]	validation_0-auc:0.95024
AUC: 0.9502


0.9502432175836142